# KeyCheck — QWERTZ Proxy Pipeline (Notebook เดียว)
Kaggle QWERTZ → Audit/Split → Rectify → OCR → Train Detector → Pipeline eval → Proxy Test
แผน: `docs/qwertz-notebook-pipeline-plan.md` v0.3 · Logic อยู่ใน `ai/` Notebook นี้มีแค่ Config / เรียก Stage / แสดงผล

**วิธีใช้:** ตั้ง `MODE="smoke"` แล้ว Run all เพื่อทดสอบทั้งเส้นก่อน → เปลี่ยนเป็น `"full"` แล้ว Run all. Stage ที่มีผลและ Config ตรงกันจะถูกข้าม; Q4/Q5 ที่ถูกขัดจังหวะจะ Resume เอง

## §0 Config

In [ ]:
ENV           = "local"                          # "local" (RTX 4060) หรือ "colab"
DATA_ROOT     = "datasets/dataset_final"         # Colab: /content/data/dataset_final
OUTPUT_ROOT   = "outputs/pipeline"               # Colab: /content/drive/MyDrive/keycheck/pipeline
PIPELINE_ID   = "qwertz_v1"
MODE          = "smoke"                          # "smoke" = 50 ต้นฉบับ + 1 epoch ; "full" = ของจริง
REPO_REF      = ""                               # commit (ขึ้นต้นก็ได้) ที่ checkout ไว้ตอนรัน Q7
SEED          = 0
RUN           = {"q1": True, "q2": True, "q3": True, "q4": True, "q5": True, "q6": True}
FORCE         = set()                            # เช่น {"q3_ocr"} = รัน Stage นั้นใหม่ (Q7 ไม่รับ FORCE)
RUN_FINAL_TEST = False                           # 🔒 Q7 รันเฉพาะเมื่อตั้งเป็น True เอง
CONFIG_FILE   = "experiments/configs/qwertz/pipeline.yaml"

## §1 Setup

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd() if (Path.cwd() / "ai").exists() else Path.cwd().parent
os.chdir(REPO); sys.path.insert(0, str(REPO))
from ai.colab import bootstrap as B
from ai.pipeline import stages as S, show

if ENV == "colab":
    B.mount_drive()
cfg  = S.load_config(CONFIG_FILE)
root = B.pipeline_root(PIPELINE_ID, MODE, OUTPUT_ROOT)
ctx  = S.Context(cfg=cfg, root=root, dataset_root=Path(DATA_ROOT), mode=MODE, seed=SEED, repo_ref=REPO_REF,
                 force=set(FORCE), run=RUN, run_final_test=RUN_FINAL_TEST)
B.seed_everything(SEED)
info = B.write_run_info(root, repo_dir=REPO, seed=SEED, mode=MODE, extra={"env": ENV})
print("output :", root); print("mode   :", MODE, "| commit:", info["commit"][:10], "| dirty:", info["dirty"])
print("GPU    :", info["nvidia_smi"] or "ไม่พบ nvidia-smi")
assert ctx.dataset_root.exists(), f"ไม่พบ dataset ที่ {ctx.dataset_root}"

## §2 Q1 — Ingest + Audit + Split

In [ ]:
r1 = S.q1(ctx)
show.show_q1(r1, ctx.dataset_root)

## §3 Q2 — Rectified dataset + S1–S4
ตรวจด้วยตา: กรอบสีเขียวต้องตรงปุ่ม · ภาพ S3 ต้องสลับปุ่มตามที่ระบุ

In [ ]:
r2 = S.q2(ctx)
show.show_q2(r2)

## §4 Q3 — OCR eval + เลือก `px_per_unit`

In [ ]:
r3 = S.q3(ctx)
show.show_q3(r3)

## §5 Q4 — Train YOLO11n (GPU, Resume ได้)

In [ ]:
r4 = S.q4(ctx)
if r4: show.show_train(r4)

## §6 Q5 — Train Faster R-CNN (GPU, Resume ได้)
ปิดได้ที่ `RUN["q5"] = False`

In [ ]:
r5 = S.q5(ctx)
if r5: show.show_train(r5)

## §7 Q6 — Pipeline eval บน Validation + Dev bundle
หลังจบ: Commit `experiments/configs/qwertz/thresholds.yaml` และ `data/manifests/kaggle_qwertz_v1/frozen_hashes.json` (และ `split_manifest.json`) ก่อนรัน Q7

In [ ]:
r6 = S.q6(ctx)
show.show_q6(r6)

## §8 Q7 — Proxy Test 🔒
รันครั้งเดียวเมื่อผ่านเงื่อนไขทั้งหมด (`RUN_FINAL_TEST=True`, `MODE="full"`, repo สะอาด, Hash ตรงกับที่ Commit). ไม่รับ FORCE

In [ ]:
r7 = S.q7(ctx)

## §9 Summary

In [ ]:
show.show_summary(ctx)